# ML day trader: variant search on a Kaggle GPU

1. On your PC run `.venv/bin/python -m hft.ml.package` and upload `dist/kaggle/hft-ml-data.zip` and
   `dist/kaggle/hft-ml-code.zip` as two private Kaggle datasets.
2. Attach both datasets to this notebook, set **Accelerator: GPU** (phone-verified account) and **Run all**.
3. Download `search-results.zip` from the output panel and unzip it into the repository's `artifacts/`.

Scores use the validation years only. The final test is run once, later, on your PC from the
saved frozen models: `python -m hft.ml.search --out artifacts/ml-search --final`. It records
itself and refuses to run twice.

In [ ]:
import sys, json, subprocess
from pathlib import Path

inputs = Path("/kaggle/input")
code_dir = next(p.parent.parent.parent for p in inputs.rglob("hft/ml/search.py"))
data_dir = next(p.parent for p in inputs.rglob("calendar.json"))
sys.path.insert(0, str(code_dir))
import torch, lightgbm, numpy, pyarrow, scipy
print("code:", code_dir, "data:", data_dir)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from hft.ml.search import grid, run_search

device = "cuda" if torch.cuda.is_available() else "cpu"
configs = grid(device=device, threads=4)
print(len(configs), "variants")
summary = run_search(configs, data_dir, "/kaggle/working/search")
print(json.dumps(summary, indent=2))

In [ ]:
subprocess.run(["zip", "-qr", "/kaggle/working/search-results.zip", "search"], cwd="/kaggle/working", check=True)
print("download /kaggle/working/search-results.zip")